# 01. Favorita 판매 데이터 탐색

판매 규모와 시간 패턴, 프로모션 효과를 확인합니다. 모델링에 사용할 수 없는 미래 정보는 만들지 않습니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw"

In [ ]:
def load_data(raw_path):
    train = pd.read_csv(
        raw_path / "train.csv",
        usecols=["date", "store_nbr", "family", "sales", "onpromotion"],
        parse_dates=["date"],
        dtype={"store_nbr": "int16", "family": "category", "sales": "float32", "onpromotion": "int16"},
    )
    stores = pd.read_csv(raw_path / "stores.csv")
    transactions = pd.read_csv(raw_path / "transactions.csv", parse_dates=["date"])
    return train, stores, transactions


def data_summary(frame):
    return pd.DataFrame({
        "value": [
            len(frame),
            frame["date"].min().date(),
            frame["date"].max().date(),
            frame["store_nbr"].nunique(),
            frame["family"].nunique(),
            frame["sales"].isna().sum(),
            frame["onpromotion"].isna().sum(),
        ]
    }, index=["rows", "start_date", "end_date", "stores", "families", "missing_sales", "missing_promotion"])


def daily_sales(frame):
    return frame.groupby("date", as_index=False, observed=True)["sales"].sum()


def promotion_summary(frame):
    return (
        frame.assign(promoted=frame["onpromotion"].gt(0))
        .groupby("promoted", observed=True)["sales"]
        .agg(rows="size", mean="mean", median="median")
    )

In [ ]:
train, stores, transactions = load_data(RAW_PATH)
data_summary(train)

## 판매량 분포

판매량은 오른쪽 꼬리가 길기 때문에 원자료와 `log1p` 변환 결과를 함께 확인합니다.

In [ ]:
sample = train["sales"].sample(min(300_000, len(train)), random_state=42).clip(lower=0)
zero_rate = train["sales"].eq(0).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sample.clip(upper=sample.quantile(0.99)), bins=60)
axes[1].hist(np.log1p(sample), bins=60)
axes[0].set(title=f"Sales distribution below 99th percentile | zero={zero_rate:.1%}", xlabel="sales")
axes[1].set(title="Log-transformed sales", xlabel="log1p(sales)")
plt.tight_layout()

## 전체 판매 추이와 요일 패턴

In [ ]:
daily = daily_sales(train)
daily["rolling_28d"] = daily["sales"].rolling(28, min_periods=1).mean()
daily["dayofweek"] = daily["date"].dt.day_name().str[:3]
weekday_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(daily["date"], daily["sales"], alpha=0.25, linewidth=0.7)
axes[0].plot(daily["date"], daily["rolling_28d"], linewidth=1.5)
axes[0].set(title="Daily sales and 28-day moving average", xlabel="", ylabel="sales")
weekday_sales = [daily.loc[daily["dayofweek"].eq(day), "sales"] for day in weekday_order]
axes[1].boxplot(weekday_sales, tick_labels=weekday_order, showfliers=False)
axes[1].set(title="Daily sales by weekday", xlabel="", ylabel="sales")
plt.tight_layout()

## 상품군과 점포별 판매 규모

In [ ]:
family_sales = (
    train.groupby("family", observed=True)["sales"].sum()
    .sort_values(ascending=False).head(15).sort_values()
)
store_sales = (
    train.groupby("store_nbr", observed=True)["sales"].sum()
    .sort_values(ascending=False).head(15).sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
family_sales.plot.barh(ax=axes[0], title="Top product families")
store_sales.plot.barh(ax=axes[1], title="Top stores")
axes[0].set(xlabel="total sales", ylabel="")
axes[1].set(xlabel="total sales", ylabel="store")
plt.tight_layout()

## 프로모션과 판매량

아래 차이는 인과효과가 아니라 단순 집단 비교입니다. 상품군과 시점이 다르므로 모델 변수의 필요성을 판단하는 탐색 결과로만 사용합니다.

In [ ]:
promotion_summary(train)

In [ ]:
promotion_by_family = (
    train.assign(promoted=train["onpromotion"].gt(0))
    .groupby(["family", "promoted"], observed=True)["sales"].mean()
    .unstack()
    .dropna()
)
promotion_by_family["mean_difference"] = promotion_by_family[True] - promotion_by_family[False]
promotion_by_family.sort_values("mean_difference", ascending=False).head(15)

## 1차 확인 사항

- 판매량의 치우침과 0 판매 비율을 고려해 RMSLE를 주 지표로 사용합니다.
- 요일과 장기 추세가 보여 달력 변수와 과거 판매량 변수가 필요합니다.
- 프로모션 집단의 차이는 크지만 인과관계로 해석하지 않습니다.
- 상품군과 점포 간 판매 규모가 달라 범주형 변수를 유지합니다.